In [36]:
!pip install pyspark

In [37]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("HospitalW2_MonitoreoPacientes")
    .getOrCreate()
)
print("Version de Apache Spark:",spark.version)
print("Sesion de Spark iniciada correctamente")

Version de Apache Spark: 4.0.4
Sesion de Spark iniciada correctamente


In [38]:
import csv
import random
from datetime import datetime,timedelta

random.seed(42)

ARCHIVO_CSV = "/content/logs_monitoreo_pacientes.csv"
TOTAL_REGISTROS = 6001
TOTAL_PACIENTES = 8

fecha_inicio = datetime(2026,1,1,0,0,0)

with open(ARCHIVO_CSV, mode="w", newline="",encoding="utf-8") as archivo:
  escritor = csv.writer(archivo)

  escritor.writerow([
      "ID_Registro",
      "ID_Paciente",
      "Tiemstamp",
      "Frecuencia_Cardiaca",
      "Saturacion_Oxigeno",
      "Alerta_Generada"
  ])
  for registro_id in range(1, TOTAL_REGISTROS + 1):

        paciente_id = random.randint(1, TOTAL_PACIENTES)

        timestamp = fecha_inicio + timedelta(
            minutes=random.randint(0, 180 * 24 * 60)
        )

        # Valores sintéticos de frecuencia cardiaca
        frecuencia = random.randint(60, 100)

        # Valores sintéticos de saturación de oxígeno
        saturacion = random.randint(92, 100)

        # Introducir eventos críticos para probar el análisis
        if random.random() < 0.12:
            saturacion = random.randint(82, 89)

        # Introducir anomalías intencionales
        if random.random() < 0.01:
            frecuencia = random.choice([-5, 0, 350])

        if random.random() < 0.01:
            saturacion = random.choice([-1, 0, 105])

        # Introducir valores nulos para probar la limpieza
        if random.random() < 0.02:
            frecuencia = ""

        if random.random() < 0.02:
            saturacion = ""

        # Regla sintética de generación de alertas
        if saturacion != "" and 0 <= saturacion < 90:
            alerta = "Si"
        elif saturacion == "" or frecuencia == "":
            alerta = "Sin dato"
        else:
            alerta = "No"

        escritor.writerow([
            registro_id,
            paciente_id,
            timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            frecuencia,
            saturacion,
            alerta
        ])

print("Archivo generado:", ARCHIVO_CSV)
print("Registros generados:", TOTAL_REGISTROS)



Archivo generado: /content/logs_monitoreo_pacientes.csv
Registros generados: 6001


In [39]:
import os
import csv

print("¿Existe el archivo?", os.path.exists(ARCHIVO_CSV))
print("Tamaño del archivo:", os.path.getsize(ARCHIVO_CSV), "bytes")

with open(ARCHIVO_CSV, mode="r", encoding="utf-8") as archivo:
    lector = csv.reader(archivo)

    encabezados = next(lector)
    primeras_filas = [next(lector) for _ in range(5)]

print("\nColumnas:")
print(encabezados)

print("\nPrimeras 5 filas:")
for fila in primeras_filas:
    print(fila)

¿Existe el archivo? True
Tamaño del archivo: 222545 bytes

Columnas:
['ID_Registro', 'ID_Paciente', 'Tiemstamp', 'Frecuencia_Cardiaca', 'Saturacion_Oxigeno', 'Alerta_Generada']

Primeras 5 filas:
['1', '2', '2026-01-05 13:16:00', '77', '95', 'No']
['2', '7', '2026-01-06 18:51:00', '61', '93', 'No']
['3', '7', '2026-02-10 03:07:00', '-5', '96', 'No']
['4', '6', '2026-01-19 14:33:00', '65', '87', 'Si']
['5', '7', '2026-01-15 08:16:00', '95', '96', 'No']


In [40]:
from google.colab import files
files.download(ARCHIVO_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
from pyspark.sql import functions as F

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(ARCHIVO_CSV)
)

print("CSV cargado correctamente.")
print("Columnas detectadas:", df_raw.columns)

CSV cargado correctamente.
Columnas detectadas: ['ID_Registro', 'ID_Paciente', 'Tiemstamp', 'Frecuencia_Cardiaca', 'Saturacion_Oxigeno', 'Alerta_Generada']


In [42]:
df_raw.show(10,truncate=False)

+-----------+-----------+-------------------+-------------------+------------------+---------------+
|ID_Registro|ID_Paciente|Tiemstamp          |Frecuencia_Cardiaca|Saturacion_Oxigeno|Alerta_Generada|
+-----------+-----------+-------------------+-------------------+------------------+---------------+
|1          |2          |2026-01-05 13:16:00|77                 |95                |No             |
|2          |7          |2026-01-06 18:51:00|61                 |93                |No             |
|3          |7          |2026-02-10 03:07:00|-5                 |96                |No             |
|4          |6          |2026-01-19 14:33:00|65                 |87                |Si             |
|5          |7          |2026-01-15 08:16:00|95                 |96                |No             |
|6          |1          |2026-05-01 09:06:00|74                 |96                |No             |
|7          |6          |2026-01-30 14:38:00|83                 |97                |No     

In [43]:
print("Esquema del DataFrame: ")
df_raw.printSchema()

total_registros = df_raw.count()

print("Total de rregistros:",total_registros)
print("Total de columnas:",len(df_raw.columns))

Esquema del DataFrame: 
root
 |-- ID_Registro: string (nullable = true)
 |-- ID_Paciente: string (nullable = true)
 |-- Tiemstamp: string (nullable = true)
 |-- Frecuencia_Cardiaca: string (nullable = true)
 |-- Saturacion_Oxigeno: string (nullable = true)
 |-- Alerta_Generada: string (nullable = true)

Total de rregistros: 6001
Total de columnas: 6


In [44]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Asegurar que la sesión de Spark existe
spark = SparkSession.builder.getOrCreate()

# Asegurar que el DataFrame df_raw esté cargado en la sesión de ejecución actual
ARCHIVO_CSV = "/content/logs_monitoreo_pacientes.csv"
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(ARCHIVO_CSV)
)
print("df_raw inicializado correctamente y listo para usarse.")

df_raw inicializado correctamente y listo para usarse.


In [45]:
columnas = df_raw.columns

expresiones_nulos = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c))==""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in columnas
]

df_nulos = df_raw.select(*expresiones_nulos)

print("Cantidad de valores nulos o vacios por columna:")
df_nulos.show(truncate=False)

Cantidad de valores nulos o vacios por columna:
+-----------+-----------+---------+-------------------+------------------+---------------+
|ID_Registro|ID_Paciente|Tiemstamp|Frecuencia_Cardiaca|Saturacion_Oxigeno|Alerta_Generada|
+-----------+-----------+---------+-------------------+------------------+---------------+
|0          |0          |0        |98                 |135               |0              |
+-----------+-----------+---------+-------------------+------------------+---------------+



In [46]:
df_exploracion = (
    df_raw
    .withColumn(
        "Frecuencia_Numerica",
        F.expr("try_cast(Frecuencia_Cardiaca AS INT)")
    )
    .withColumn(
        "Saturacion_Numerica",
        F.expr("try_cast(Saturacion_Oxigeno AS INT)")
    )
)

print("Estadísticas descriptivas de las mediciones:")
df_exploracion.select(
    F.min("Frecuencia_Numerica").alias("Frecuencia_Minima"),
    F.max("Frecuencia_Numerica").alias("Frecuencia_Maxima"),
    F.avg("Frecuencia_Numerica").alias("Frecuencia_Promedio"),
    F.min("Saturacion_Numerica").alias("Saturacion_Minima"),
    F.max("Saturacion_Numerica").alias("Saturacion_Maxima"),
    F.avg("Saturacion_Numerica").alias("Saturacion_Promedio")
).show(truncate=False)

Estadísticas descriptivas de las mediciones:
+-----------------+-----------------+-------------------+-----------------+-----------------+-------------------+
|Frecuencia_Minima|Frecuencia_Maxima|Frecuencia_Promedio|Saturacion_Minima|Saturacion_Maxima|Saturacion_Promedio|
+-----------------+-----------------+-------------------+-----------------+-----------------+-------------------+
|-5               |350              |80.46823648992039  |-1               |105              |94.10040913740198  |
+-----------------+-----------------+-------------------+-----------------+-----------------+-------------------+



In [47]:
frecuencia_invalida = (
    F.col("Frecuencia_Numerica").isNull()
    | ~F.col("Frecuencia_Numerica").between(30, 220)
)

saturacion_invalida = (
    F.col("Saturacion_Numerica").isNull()
    | ~F.col("Saturacion_Numerica").between(70, 100)
)

resumen_anomalias = df_exploracion.select(
    F.sum(
        F.when(frecuencia_invalida, 1).otherwise(0)
    ).alias("FrecuenciasInvalidas"),
    F.sum(
        F.when(saturacion_invalida, 1).otherwise(0)
    ).alias("SaturacionesInvalidas"),
    F.sum(
        F.when(
            F.col("Saturacion_Numerica").between(0, 89),
            1
        ).otherwise(0)
    ).alias("SaturacionesMenores90")
)

resumen_anomalias.show(truncate=False)

+--------------------+---------------------+---------------------+
|FrecuenciasInvalidas|SaturacionesInvalidas|SaturacionesMenores90|
+--------------------+---------------------+---------------------+
|163                 |193                  |690                  |
+--------------------+---------------------+---------------------+



In [48]:
# Revisar identificadores de registro duplicados
from pyspark.sql import functions as F

duplicados =(
    df_raw.groupBy("ID_Registro")
    .count()
    .filter(F.col("count")>1)
)
print("Identificadores de registro duplicados: ")
duplicados.show(20,truncate=False)

#Revisar fechas que no pueden interpretarse (Corregido 'Timestamp' por 'Tiemstamp')
df_fechas = df_raw.withColumn(
    "Fecha_Convertida",
    F.to_timestamp(F.col("Tiemstamp"),
                   "yyyy-MM-dd HH:mm:ss")
)

print("Fechas interpretables: ")
df_fechas.filter(F.col("Fecha_Convertida").isNull()
).select("ID_Registro","Tiemstamp").show(20, truncate = False)

#Verificar pacientes registrados en el csv
print("Identificadores de paciente encontrados: ")

df_raw.select("ID_Paciente").distinct().orderBy("ID_Paciente").show()

Identificadores de registro duplicados: 
+-----------+-----+
|ID_Registro|count|
+-----------+-----+
+-----------+-----+

Fechas interpretables: 
+-----------+---------+
|ID_Registro|Tiemstamp|
+-----------+---------+
+-----------+---------+

Identificadores de paciente encontrados: 
+-----------+
|ID_Paciente|
+-----------+
|          1|
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
+-----------+



In [49]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType
)

df_tipos = (
    df_raw
    .withColumn(
        "ID_Registro",
        F.expr("try_cast(ID_Registro AS INT)")
    )
    .withColumn(
        "ID_Paciente",
        F.expr("try_cast(ID_Paciente AS INT)")
    )
    .withColumn(
        "Timestamp",
        F.to_timestamp("Tiemstamp", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "Frecuencia_Cardiaca",
        F.expr("try_cast(Frecuencia_Cardiaca AS INT)")
    )
    .withColumn(
        "Saturacion_Oxigeno",
        F.expr("try_cast(Saturacion_Oxigeno AS INT)")
    )
    .withColumn(
        "Alerta_Generada",
        F.trim(F.col("Alerta_Generada"))
    )
)

# Eliminamos la columna original con el error de escritura para evitar duplicidad
df_tipos = df_tipos.drop("Tiemstamp")

df_tipos.printSchema()
df_tipos.show(5, truncate=False)

root
 |-- ID_Registro: integer (nullable = true)
 |-- ID_Paciente: integer (nullable = true)
 |-- Frecuencia_Cardiaca: integer (nullable = true)
 |-- Saturacion_Oxigeno: integer (nullable = true)
 |-- Alerta_Generada: string (nullable = true)
 |-- Timestamp: timestamp (nullable = true)

+-----------+-----------+-------------------+------------------+---------------+-------------------+
|ID_Registro|ID_Paciente|Frecuencia_Cardiaca|Saturacion_Oxigeno|Alerta_Generada|Timestamp          |
+-----------+-----------+-------------------+------------------+---------------+-------------------+
|1          |2          |77                 |95                |No             |2026-01-05 13:16:00|
|2          |7          |61                 |93                |No             |2026-01-06 18:51:00|
|3          |7          |-5                 |96                |No             |2026-02-10 03:07:00|
|4          |6          |65                 |87                |Si             |2026-01-19 14:33:00|
|5   

In [50]:
total_original = df_tipos.count()

df_limpio = df_tipos.filter(
    F.col("ID_Registro").isNotNull()
    & F.col("ID_Paciente").isNotNull()
    & F.col("Timestamp").isNotNull()
    & F.col("Frecuencia_Cardiaca").between(30, 220)
    & F.col("Saturacion_Oxigeno").between(70, 100)
    & F.col("Alerta_Generada").isNotNull()
    & (F.col("Alerta_Generada") != "")
)

# Comprobar que los identificadores no estén duplicados.
# Si hubiera duplicados, conservaremos una fila por ID.
df_limpio = df_limpio.dropDuplicates(["ID_Registro"])

total_limpio = df_limpio.count()
total_excluido = total_original - total_limpio

print("Registros originales:", total_original)
print("Registros después de limpieza:", total_limpio)
print("Registros excluidos o duplicados:", total_excluido)

Registros originales: 6001
Registros después de limpieza: 5649
Registros excluidos o duplicados: 352


In [51]:
from pyspark.sql.functions import udf

def clasificar_riesgo(saturacion, frecuencia):
    if saturacion is None or frecuencia is None:
        return "Moderado"

    if saturacion < 90 or frecuencia < 50 or frecuencia > 120:
        return "Crítico"

    if saturacion <= 94 or frecuencia < 60 or frecuencia > 100:
        return "Moderado"

    return "Bajo"


Nivel_Riesgo = udf(clasificar_riesgo, StringType())

df_procesado = df_limpio.withColumn(
    "Nivel_Riesgo",
    Nivel_Riesgo(
        F.col("Saturacion_Oxigeno"),
        F.col("Frecuencia_Cardiaca")
    )
)

df_procesado.groupBy("Nivel_Riesgo").count().show()

+------------+-----+
|Nivel_Riesgo|count|
+------------+-----+
|     Crítico|  651|
|    Moderado| 1674|
|        Bajo| 3324|
+------------+-----+



In [52]:
df_alertas_criticas = df_procesado.filter(
    F.col("Saturacion_Oxigeno") < 90
)

top5_pacientes = (
    df_alertas_criticas
    .groupBy("ID_Paciente")
    .agg(
        F.count("*").alias("Total_Alertas_Criticas")
    )
    .orderBy(
        F.desc("Total_Alertas_Criticas"),
        F.asc("ID_Paciente")
    )
    .limit(5)
)

print("Top 5 de pacientes con saturación inferior al 90 %:")
top5_pacientes.show(truncate=False)

Top 5 de pacientes con saturación inferior al 90 %:
+-----------+----------------------+
|ID_Paciente|Total_Alertas_Criticas|
+-----------+----------------------+
|8          |94                    |
|2          |88                    |
|5          |85                    |
|3          |80                    |
|6          |79                    |
+-----------+----------------------+



In [53]:
print("Distribución de niveles de riesgo:")
df_procesado.groupBy("Nivel_Riesgo").count().orderBy(
    "Nivel_Riesgo"
).show()

Distribución de niveles de riesgo:
+------------+-----+
|Nivel_Riesgo|count|
+------------+-----+
|        Bajo| 3324|
|     Crítico|  651|
|    Moderado| 1674|
+------------+-----+



In [54]:
df_comparacion = df_procesado.withColumn(
    "Alerta_SpO2_Critica",
    F.when(F.col("Saturacion_Oxigeno") < 90, "Si")
     .otherwise("No")
)

df_comparacion.groupBy(
    "Alerta_Generada",
    "Alerta_SpO2_Critica"
).count().orderBy(
    "Alerta_Generada",
    "Alerta_SpO2_Critica"
).show(truncate=False)

+---------------+-------------------+-----+
|Alerta_Generada|Alerta_SpO2_Critica|count|
+---------------+-------------------+-----+
|No             |No                 |4998 |
|Si             |Si                 |651  |
+---------------+-------------------+-----+



In [55]:
RUTA_PARQUET = "/content/HospitalW2_Parquet"

(
    df_procesado
    .write
    .mode("overwrite")
    .parquet(RUTA_PARQUET)
)

print("Datos exportados a:", RUTA_PARQUET)

Datos exportados a: /content/HospitalW2_Parquet


In [56]:
df_parquet = spark.read.parquet(RUTA_PARQUET)

print("Registros procesados:", df_procesado.count())
print("Registros recuperados:", df_parquet.count())

print("\nEsquema del Parquet:")
df_parquet.printSchema()

print("\nMuestra de datos exportados:")
df_parquet.show(10, truncate=False)

Registros procesados: 5649
Registros recuperados: 5649

Esquema del Parquet:
root
 |-- ID_Registro: integer (nullable = true)
 |-- ID_Paciente: integer (nullable = true)
 |-- Frecuencia_Cardiaca: integer (nullable = true)
 |-- Saturacion_Oxigeno: integer (nullable = true)
 |-- Alerta_Generada: string (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Nivel_Riesgo: string (nullable = true)


Muestra de datos exportados:
+-----------+-----------+-------------------+------------------+---------------+-------------------+------------+
|ID_Registro|ID_Paciente|Frecuencia_Cardiaca|Saturacion_Oxigeno|Alerta_Generada|Timestamp          |Nivel_Riesgo|
+-----------+-----------+-------------------+------------------+---------------+-------------------+------------+
|1          |2          |77                 |95                |No             |2026-01-05 13:16:00|Bajo        |
|2          |7          |61                 |93                |No             |2026-01-06 18:51:00|Moder

In [57]:
import os
import platform
import subprocess

print("Sistema operativo:", platform.system())
print("Versión de Python:", platform.python_version())

resultado = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print("Java:")
print(resultado.stderr or resultado.stdout)

Sistema operativo: Linux
Versión de Python: 3.13.16
Java:
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)



In [58]:
import subprocess
import time
import os
import getpass

# Instalar Tailscale en el entorno Linux de Colab
subprocess.run(
    ["bash", "-lc", "curl -fsSL https://tailscale.com/install.sh | sh"],
    check=True
)

# Iniciar Tailscale en modo de espacio de usuario.
# El proxy SOCKS5 solo escucha en localhost de Colab.
daemon = subprocess.Popen(
    [
        "tailscaled",
        "--tun=userspace-networking",
        "--socks5-server=127.0.0.1:1055",
        "--state=mem:"
    ],
    stdout=open("/tmp/tailscaled.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

# Solicitar la clave sin mostrarla en pantalla
auth_key = getpass.getpass("Pega tu clave temporal de Tailscale: ")

try:
    resultado = subprocess.run(
        ["tailscale", "up", f"--auth-key={auth_key}"],
        capture_output=True,
        text=True,
        timeout=60
    )
    print("Código de salida:", resultado.returncode)
    if resultado.returncode != 0:
        print("Error:", resultado.stderr[-1500:])
    else:
        print("Tailscale se ha autenticado.")
finally:
    auth_key = None

Pega tu clave temporal de Tailscale: ··········
Código de salida: 1
Error: backend error: invalid key: unable to validate API key



In [59]:
import subprocess

subprocess.run(["tailscale", "status"], check=False)
subprocess.run(["tailscale", "ip", "-4"], check=False)

CompletedProcess(args=['tailscale', 'ip', '-4'], returncode=1)

In [60]:
import subprocess

resultado = subprocess.run(
    ["tailscale", "status"],
    capture_output=True,
    text=True
)

print(resultado.stdout if resultado.returncode == 0 else resultado.stderr)

In [61]:
import ipaddress
import socket
import getpass

windows_ip = getpass.getpass(
    "Introduce la IPv4 de Tailscale de Windows: "
).strip()

# Validar que sea una dirección IPv4 privada de Tailscale
ip = ipaddress.ip_address(windows_ip)
if ip.version != 4 or not ip in ipaddress.ip_network("100.64.0.0/10"):
    raise ValueError("La dirección no pertenece al rango IPv4 de Tailscale.")

try:
    with socket.create_connection((windows_ip, 1433), timeout=5):
        print("Conexión TCP privada al puerto 1433: correcta")
except OSError as e:
    print("No se pudo conectar por TCP:", e)
finally:
    windows_ip = None

KeyboardInterrupt: Interrupted by user

In [ ]:
import subprocess

for comando in [
    ["tailscale", "status"],
    ["tailscale", "ping", "--timeout=5s", "desktop-pvjfi3m"]
]:
    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True,
        timeout=15
    )
    print("$", " ".join(comando))
    print(resultado.stdout or resultado.stderr)

In [ ]:
%pip -q install PySocks

In [ ]:
import socket
import socks

WINDOWS_IP = "100.71.218.114"

conexion = socks.socksocket()
conexion.set_proxy(
    socks.SOCKS5,
    "127.0.0.1",
    1055
)
conexion.settimeout(10)

try:
    conexion.connect((WINDOWS_IP, 1433))
    print("ÉXITO: conexión TCP a SQL Server establecida por Tailscale.")
except Exception as e:
    print(f"No se pudo establecer la conexión TCP: {type(e).__name__}: {e}")
finally:
    conexion.close()

In [ ]:
import socket
import socketserver
import select
import threading
import socks

WINDOWS_IP = "100.71.218.114"
WINDOWS_SQL_PORT = 1433

LOCAL_HOST = "127.0.0.1"
LOCAL_PORT = 11433
SOCKS_HOST = "127.0.0.1"
SOCKS_PORT = 1055


class SQLTunnelHandler(socketserver.BaseRequestHandler):
    def handle(self):
        cliente = self.request
        servidor = socks.socksocket()

        servidor.set_proxy(
            socks.SOCKS5,
            SOCKS_HOST,
            SOCKS_PORT
        )
        servidor.settimeout(15)

        try:
            servidor.connect((WINDOWS_IP, WINDOWS_SQL_PORT))
            servidor.settimeout(None)
            cliente.settimeout(None)

            conexiones = [cliente, servidor]

            while True:
                legibles, _, _ = select.select(
                    conexiones, [], [], 60
                )

                if not legibles:
                    continue

                for origen in legibles:
                    datos = origen.recv(65536)

                    if not datos:
                        return

                    destino = servidor if origen is cliente else cliente
                    destino.sendall(datos)

        except (OSError, socks.ProxyError):
            pass
        finally:
            servidor.close()


class SQLTunnelServer(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True


# Evitar iniciar dos puentes en la misma sesión
if "sql_tunnel_server" not in globals():
    sql_tunnel_server = SQLTunnelServer(
        (LOCAL_HOST, LOCAL_PORT),
        SQLTunnelHandler
    )

    sql_tunnel_thread = threading.Thread(
        target=sql_tunnel_server.serve_forever,
        daemon=True
    )
    sql_tunnel_thread.start()

    print(f"Puente activo en {LOCAL_HOST}:{LOCAL_PORT}")
else:
    print(f"El puente ya está iniciado en {LOCAL_HOST}:{LOCAL_PORT}")

In [ ]:
import socket

with socket.create_connection(
    ("127.0.0.1", 11433),
    timeout=10
):
    print("ÉXITO: el puente TCP local acepta conexiones.")

In [ ]:
import os
import urllib.request

jdbc_dir = "/content/jdbc"
os.makedirs(jdbc_dir, exist_ok=True)

jdbc_jar = os.path.join(
    jdbc_dir,
    "mssql-jdbc-12.8.1.jre11.jar"
)

if not os.path.exists(jdbc_jar):
    url = (
        "https://repo1.maven.org/maven2/com/microsoft/sqlserver/"
        "mssql-jdbc/12.8.1.jre11/mssql-jdbc-12.8.1.jre11.jar"
    )
    urllib.request.urlretrieve(url, jdbc_jar)

print("Controlador descargado:", os.path.exists(jdbc_jar))
print("Ruta:", jdbc_jar)

In [ ]:
import os
import socket

jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"

print("Controlador JDBC disponible:", os.path.isfile(jdbc_jar))

try:
    with socket.create_connection(("127.0.0.1", 11433), timeout=5):
        print("Puente TCP local: responde")
except Exception as e:
    print("Puente TCP local: error:", str(e))

In [ ]:
import os
import socket

jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"

print("JAR JDBC disponible:", os.path.isfile(jdbc_jar))

try:
    with socket.create_connection(("127.0.0.1", 11433), timeout=5):
        print("Puente TCP local: OK")
except Exception as e:
    print("Puente TCP local: ERROR:", str(e))

In [ ]:
print("Versión de Spark:", spark.version)
print("Sesión activa:", not spark.sparkContext._jsc.sc().isStopped())

In [ ]:
import socket

try:
    with socket.create_connection(("127.0.0.1", 11433), timeout=5):
        print("Puente TCP local: OK")
except Exception as e:
    print("Error en el puente TCP:", e)

In [ ]:
jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"

assert __import__("os").path.isfile(jdbc_jar), "No se encuentra el JAR JDBC"

spark.sparkContext._jsc.addJar(jdbc_jar)

print("JAR agregado a Spark.")

In [ ]:
from getpass import getpass

sql_user = "hospital_pyspark"
sql_password = getpass("Contraseña de hospital_pyspark: ")

jdbc_url = (
    "jdbc:sqlserver://127.0.0.1:11433;"
    "databaseName=HospitalW2;"
    "encrypt=true;"
    "trustServerCertificate=true;"
    "loginTimeout=15;"
)

print("Configuración preparada para HospitalW2.")

In [ ]:
import os
import zipfile
import subprocess

jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"

print("=== ARCHIVO JDBC ===")
print("Existe:", os.path.isfile(jdbc_jar))

if os.path.isfile(jdbc_jar):
    print("Tamaño:", os.path.getsize(jdbc_jar), "bytes")
    try:
        with zipfile.ZipFile(jdbc_jar, "r") as jar:
            clase = "com/microsoft/sqlserver/jdbc/SQLServerDriver.class"
            print("JAR válido: Sí")
            print("Clase del controlador presente:", clase in jar.namelist())
    except Exception as e:
        print("Error al leer el JAR:", repr(e))

print("\n=== JAVA ===")
try:
    resultado = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print(resultado.stderr or resultado.stdout)
except Exception as e:
    print("Error:", repr(e))

print("\n=== SPARK ===")
try:
    print("Versión:", spark.version)
    print("Contexto detenido:", spark.sparkContext._jsc.sc().isStopped())
    print("JAR registrado:", jdbc_jar in list(spark.sparkContext._jsc.sc().listJars()))
except Exception as e:
    print("Error al consultar Spark:", repr(e))

print("\n=== ERROR DEL CARGADOR DE CLASES ===")
try:
    jvm = spark.sparkContext._jvm
    jvm.java.lang.Class.forName(
        "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    )
    print("El controlador ya está visible para la JVM.")
except Exception as e:
    print(repr(e))

In [ ]:
import os

jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"
assert os.path.isfile(jdbc_jar), "No se encuentra el JAR JDBC"

jvm = spark.sparkContext._jvm

# Preparar la URL del archivo JAR
jar_url = jvm.java.io.File(jdbc_jar).toURI().toURL()

# OBTENER la clase java.net.URL de forma correcta para la JVM usando Class.forName
clase_url = jvm.java.lang.Class.forName("java.net.URL")

# Crear el arreglo Java URL[] usando la clase real de la JVM
urls = jvm.java.lang.reflect.Array.newInstance(clase_url, 1)
jvm.java.lang.reflect.Array.set(urls, 0, jar_url)

# Obtener el cargador compatible con Spark de forma segura
hilo = jvm.java.lang.Thread.currentThread()
cargador_padre = hilo.getContextClassLoader()

cargador = jvm.org.apache.spark.util.MutableURLClassLoader(
    urls, cargador_padre
)

# Establecer el cargador para el hilo actual
hilo.setContextClassLoader(cargador)

# Verificar la carga de la clase del Driver de SQL Server usando el nuevo cargador
controlador = jvm.java.lang.Class.forName(
    "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    True,
    cargador
)

print("ÉXITO: controlador JDBC cargado en la JVM de manera dinámica.")

In [ ]:
try:
    print("Versión de Spark:", spark.version)
    print("SparkContext activo:", spark.sparkContext._jsc.sc().isStopped() == False)
except Exception as e:
    print("Spark no está disponible:", str(e)[:500])

In [ ]:
import os

jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"

if not os.path.isfile(jdbc_jar):
    raise FileNotFoundError(f"No se encontró el archivo: {jdbc_jar}")

jvm = spark.sparkContext._jvm

# Obtener la ruta del JAR como URL de Java
jar_url = jvm.java.io.File(jdbc_jar).toURI().toURL()

# OBTENER la clase java.net.URL de forma correcta para la JVM usando Class.forName
clase_url = jvm.java.lang.Class.forName("java.net.URL")

# Crear el arreglo Java URL[] usando la clase real de la JVM en lugar del objeto Python
urls = jvm.java.lang.reflect.Array.newInstance(clase_url, 1)
jvm.java.lang.reflect.Array.set(urls, 0, jar_url)

hilo = jvm.java.lang.Thread.currentThread()
cargador_padre = hilo.getContextClassLoader()

cargador = jvm.org.apache.spark.util.MutableURLClassLoader(
    urls, cargador_padre
)

hilo.setContextClassLoader(cargador)

# Intentar cargar el controlador
controlador = jvm.java.lang.Class.forName(
    "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    True,
    cargador
)

print("ÉXITO: controlador JDBC cargado correctamente.")
print("Clase:", controlador.getName())

In [ ]:
from getpass import getpass

# Datos de conexión
sql_user = "hospital_pyspark"

sql_password = getpass("Contraseña de hospital_pyspark: ")

# Modificamos los parámetros de la URL para facilitar el apretón de manos TLS/SSL
jdbc_url = (
    "jdbc:sqlserver://127.0.0.1:11433;"
    "databaseName=HospitalW2;"
    "encrypt=false;"               # Cambiado a false para descartar problemas de handshake TLS/SSL
    "trustServerCertificate=true;"
    "loginTimeout=15;"
)

print("Intentando realizar la consulta JDBC de prueba...")

try:
    # Consulta de prueba
    prueba = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", """
            SELECT
                DB_NAME() AS BaseDatos,
                SUSER_SNAME() AS UsuarioSQL,
                @@SERVERNAME AS Servidor
        """)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )
    prueba.show(truncate=False)
except Exception as e:
    print("\n--- ERROR DETECTADO ---")
    print("No se pudo completar la conexión. Verifica que el servicio SQL Server en Windows")
    print("esté configurado para escuchar en el puerto TCP 1433 estático y que acepte conexiones externas.")
    print("Detalle:", str(e)[:1000])

In [ ]:
import socket

for puerto in [11433, 1433]:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(3)

    try:
        resultado = s.connect_ex(("127.0.0.1", puerto))
        if resultado == 0:
            print(f"Puerto {puerto}: acepta conexiones TCP en localhost")
        else:
            print(f"Puerto {puerto}: no acepta conexiones en localhost")
    except Exception as e:
        print(f"Puerto {puerto}: {e}")
    finally:
        s.close()

In [ ]:
!ss -ltnp | grep -E '11433|1055'

In [ ]:
import socket

for puerto in [1055, 11433]:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(3)
        resultado = s.connect_ex(("127.0.0.1", puerto))
        print(f"Puerto {puerto}: {'ACTIVO' if resultado == 0 else 'NO DISPONIBLE'}")

In [ ]:
import socks

WINDOWS_IP = "100.71.218.114"

conexion = socks.socksocket()
conexion.set_proxy(socks.SOCKS5, "127.0.0.1", 1055)
conexion.settimeout(10)

try:
    conexion.connect((WINDOWS_IP, 1433))
    print("ÉXITO: conexión directa mediante SOCKS5 a SQL Server.")
except Exception as e:
    print("ERROR en la ruta SOCKS5:", repr(e))
finally:
    conexion.close()

In [ ]:
!pgrep -af tailscaled
!tailscale status

In [ ]:
import subprocess
import time
import os

# Comprobar la ubicación del ejecutable
resultado = subprocess.run(
    ["which", "tailscaled"],
    capture_output=True,
    text=True
)

print("tailscaled:", resultado.stdout.strip() or "No encontrado")

In [ ]:
import os
import subprocess
import time

os.makedirs("/var/run/tailscale", exist_ok=True)

# Iniciar Tailscale en modo userspace
with open("/tmp/tailscaled.log", "w") as log:
    proceso = subprocess.Popen(
        [
            "/usr/sbin/tailscaled",
            "--tun=userspace-networking",
            "--socks5-server=127.0.0.1:1055",
            "--state=mem:",
            "--socket=/var/run/tailscale/tailscaled.sock"
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

time.sleep(3)

print("PID del nuevo proceso:", proceso.pid)

In [ ]:
!tailscale --socket=/var/run/tailscale/tailscaled.sock status

In [ ]:
!cat /tmp/tailscaled.log

In [ ]:
!tailscale --socket=/var/run/tailscale/tailscaled.sock up

In [ ]:
!tailscale --socket=/var/run/tailscale/tailscaled.sock status

In [ ]:
import socket

with socket.create_connection(("127.0.0.1", 1055), timeout=3):
    print("ÉXITO: el proxy SOCKS5 está escuchando.")

In [ ]:
prueba = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", """
        SELECT
            DB_NAME() AS BaseDatos,
            SUSER_SNAME() AS UsuarioSQL,
            @@SERVERNAME AS Servidor
    """)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

prueba.show(truncate=False)

In [ ]:
from pyspark.sql import SparkSession
from getpass import getpass
import os

# 1. Asegurar que tenemos la ruta correcta del JAR
jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"
assert os.path.isfile(jdbc_jar), f"No se encuentra el JAR en {jdbc_jar}"

# 2. Detener la sesión de Spark anterior si existe para poder reconfigurarla
try:
    spark.stop()
    print("Sesión anterior de Spark detenida correctamente.")
except NameError:
    pass

# 3. Crear una nueva SparkSession que incluya de forma nativa el JAR para Driver y Ejecutores
spark = (
    SparkSession.builder
    .appName("HospitalW2_MonitoreoPacientes")
    .config("spark.jars", jdbc_jar)
    .getOrCreate()
)
print("Nueva sesión de Spark iniciada con el JAR integrado.")

# 4. Solicitar credenciales y probar conexión distribuyendo la tarea de manera correcta
sql_password = getpass("Ingresa la contraseña de 'hospital_pyspark': ")

try:
    print("Intentando conectar con Spark distribuido...")
    prueba = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", """
            SELECT
                DB_NAME() AS BaseDatos,
                SUSER_SNAME() AS UsuarioSQL,
                @@SERVERNAME AS Servidor
        """)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )
    print("¡CONEXIÓN DE PRUEBA EXITOSA EN TODO EL CLUSTER!")
    prueba.show(truncate=False)
except Exception as e:
    print("\n--- ERROR DE CONEXIÓN ---")
    print("Verifica la contraseña y la configuración de autenticación mixta en SQL Server.")
    print("Detalle del error:", str(e)[:600])

In [ ]:
tablas = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        """
        SELECT
            TABLE_SCHEMA AS Esquema,
            TABLE_NAME AS Tabla
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
        """
    )
    .option("user", sql_user)
    .option("password", sql_password)
    .option(
        "driver",
        "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    )
    .load()
)

# Aplicamos el ordenamiento usando las funciones nativas de PySpark
tablas_ordenadas = tablas.orderBy("Esquema", "Tabla")

print("Tablas disponibles en HospitalW2:")
tablas_ordenadas.show(100, truncate=False)

In [ ]:
from pyspark.sql import functions as F

tablas_objetivo = [
    "Paciente",
    "REGISTRO_TELEMETRIA",
    "ADMISION",
    "Hospitalizacion",
    "Cita",
    "Insumo",
    "MOVIMIENTO_INVENTARIO"
]

consulta_columnas = """
SELECT
    TABLE_SCHEMA AS Esquema,
    TABLE_NAME AS Tabla,
    ORDINAL_POSITION AS Posicion,
    COLUMN_NAME AS Columna,
    DATA_TYPE AS TipoDato,
    IS_NULLABLE AS AceptaNulos
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'dbo'
  AND TABLE_NAME IN (
      'Paciente',
      'REGISTRO_TELEMETRIA',
      'ADMISION',
      'Hospitalizacion',
      'Cita',
      'Insumo',
      'MOVIMIENTO_INVENTARIO'
  )
"""

columnas_hospital = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", consulta_columnas)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

# Ordenamos de forma segura dentro de PySpark
columnas_ordenadas = columnas_hospital.orderBy("Tabla", "Posicion")

columnas_ordenadas.show(200, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Leer la tabla de telemetría desde SQL Server
df_telemetria = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "dbtable",
        "dbo.REGISTRO_TELEMETRIA"
    )
    .option("user", sql_user)
    .option("password", sql_password)
    .option(
        "driver",
        "com.microsoft.sqlserver.jdbc.SQLServerDriver"
    )
    .load()
)

print("Esquema de la telemetría:")
df_telemetria.printSchema()

print("Total de registros originales:", df_telemetria.count())

df_telemetria.show(10, truncate=False)

In [ ]:
# Normalizar los tipos de datos
df_tipado = (
    df_telemetria
    .withColumn("IdRegistro", F.col("IdRegistro").cast("int"))
    .withColumn("IdPaciente", F.col("IdPaciente").cast("int"))
    .withColumn("MarcaTiempo", F.col("MarcaTiempo").cast("timestamp"))
    .withColumn(
        "FrecuenciaCardiaca",
        F.col("FrecuenciaCardiaca").cast("int")
    )
    .withColumn(
        "SaturacionOxigeno",
        F.col("SaturacionOxigeno").cast("double")
    )
)

# Eliminar duplicados por identificador
df_sin_duplicados = df_tipado.dropDuplicates(["IdRegistro"])

# Excluir registros incompletos y mediciones fuera de los rangos definidos
df_limpio = df_sin_duplicados.filter(
    F.col("IdRegistro").isNotNull()
    & F.col("IdPaciente").isNotNull()
    & F.col("MarcaTiempo").isNotNull()
    & F.col("FrecuenciaCardiaca").between(30, 220)
    & F.col("SaturacionOxigeno").between(70, 100)
)

print("Registros originales:", df_telemetria.count())
print("Después de deduplicar:", df_sin_duplicados.count())
print("Después de validar:", df_limpio.count())
print("Registros descartados en total:",
      df_telemetria.count() - df_limpio.count())

In [ ]:
condicion_critica = (
    (F.col("SaturacionOxigeno") < 90)
    | (F.col("FrecuenciaCardiaca") < 50)
    | (F.col("FrecuenciaCardiaca") > 120)
)

condicion_moderada = (
    F.col("SaturacionOxigeno").between(90, 94.999999)
    | F.col("FrecuenciaCardiaca").between(50, 59)
    | F.col("FrecuenciaCardiaca").between(101, 120)
)

df_procesado = (
    df_limpio
    .withColumn(
        "Nivel_Riesgo",
        F.when(condicion_critica, F.lit("Crítico"))
         .when(condicion_moderada, F.lit("Moderado"))
         .otherwise(F.lit("Bajo"))
    )
    .withColumn(
        "AlertaSpO2",
        F.when(F.col("SaturacionOxigeno") < 90, True)
         .otherwise(False)
    )
)

print("Distribución por nivel de riesgo:")
df_procesado.groupBy("Nivel_Riesgo").count().show()

print("Registros con SpO2 inferior al 90 %:")
df_procesado.filter(F.col("AlertaSpO2")).show(
    20, truncate=False
)

In [ ]:
ventana_paciente = Window.partitionBy("IdPaciente").orderBy(
    F.col("MarcaTiempo").desc(),
    F.col("IdRegistro").desc()
)

df_top5_paciente = (
    df_procesado
    .withColumn(
        "PosicionReciente",
        F.row_number().over(ventana_paciente)
    )
    .filter(F.col("PosicionReciente") <= 5)
)

df_top5_paciente.orderBy(
    "IdPaciente", "PosicionReciente"
).show(100, truncate=False)

In [ ]:
ruta_parquet = "/content/HospitalW2_Parquet"

(
    df_procesado.write
    .mode("overwrite")
    .parquet(ruta_parquet)
)

# Verificar que los datos se puedan recuperar
df_verificacion = spark.read.parquet(ruta_parquet)

print("Registros procesados:", df_procesado.count())
print("Registros recuperados de Parquet:", df_verificacion.count())

df_verificacion.printSchema()

In [ ]:
from pyspark.sql import functions as F

print("1. Comparación de alertas almacenadas y riesgo calculado")

df_procesado.groupBy(
    "AlertaGenerada", "Nivel_Riesgo"
).count().orderBy(
    "AlertaGenerada", "Nivel_Riesgo"
).show(truncate=False)

print("2. Registros con alerta almacenada, pero SpO2 >= 90")

df_procesado.filter(
    (F.col("AlertaGenerada") == True)
    & (F.col("SaturacionOxigeno") >= 90)
).select(
    "IdRegistro",
    "IdPaciente",
    "MarcaTiempo",
    "FrecuenciaCardiaca",
    "SaturacionOxigeno",
    "AlertaGenerada",
    "Nivel_Riesgo",
    "AlertaSpO2"
).show(truncate=False)

print("3. Resumen de calidad")

df_procesado.agg(
    F.count("*").alias("TotalRegistros"),
    F.countDistinct("IdRegistro").alias("IdsUnicos"),
    F.sum(
        F.when(F.col("FrecuenciaCardiaca").isNull(), 1).otherwise(0)
    ).alias("FrecuenciaNula"),
    F.sum(
        F.when(F.col("SaturacionOxigeno").isNull(), 1).otherwise(0)
    ).alias("SaturacionNula"),
    F.min("MarcaTiempo").alias("FechaMasAntigua"),
    F.max("MarcaTiempo").alias("FechaMasReciente")
).show(truncate=False)

In [ ]:
consulta_reglas = """
SELECT
    tr.name AS NombreTrigger,
    tr.is_disabled AS Deshabilitado,
    OBJECT_DEFINITION(tr.object_id) AS Definicion
FROM sys.triggers AS tr
WHERE tr.parent_id = OBJECT_ID('dbo.REGISTRO_TELEMETRIA')
"""

df_reglas = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", consulta_reglas)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

df_reglas.show(truncate=False)

In [ ]:

# HospitalW2 — Paso 1: diagnóstico de la sesión existente
# Solo lectura. No modifica SQL Server ni reinicia Spark.

variables = globals()

print("=== ESTADO DE LA SESIÓN ===")
print("SparkSession disponible:", "spark" in variables)
print("DataFrame de telemetría disponible:", "df_telemetria" in variables)

if "spark" in variables:
    print("Versión de Spark:", spark.version)
    print("SparkContext activo:", not spark.sparkContext._jsc.sc().isStopped())

if "df_telemetria" in variables:
    print("\n=== TELEMETRÍA ORIGINAL ===")
    print("Cantidad de registros:", df_telemetria.count())
    print("Columnas:", df_telemetria.columns)

    df_telemetria.printSchema()
    df_telemetria.orderBy("IdRegistro").show(10, truncate=False)
else:
    print(
        "\nNo se encontró df_telemetria en la sesión actual. "
        "No se ha realizado ninguna reconexión."
    )


In [ ]:

# HospitalW2 — Paso 2: generar telemetría sintética
# No inserta ni modifica datos en SQL Server.

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# 1. Obtener identificadores reales de pacientes
if all(v in globals() for v in ["jdbc_url", "sql_user", "sql_password"]):
    df_pacientes = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "(SELECT id_paciente FROM dbo.Paciente) AS p")
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )
    pacientes = [
        int(r["id_paciente"])
        for r in df_pacientes.select("id_paciente").distinct().collect()
        if r["id_paciente"] is not None
    ]
else:
    # Alternativa segura si las variables de conexión ya no están disponibles.
    pacientes = [
        int(r["IdPaciente"])
        for r in df_telemetria.select("IdPaciente").distinct().collect()
        if r["IdPaciente"] is not None
    ]

pacientes = sorted(set(pacientes))

if len(pacientes) < 1:
    raise ValueError("No se encontraron pacientes para generar telemetría.")

print("Pacientes disponibles para los datos sintéticos:", len(pacientes))

# 2. Crear 5,000 registros con identificadores nuevos.
# Los IDs empiezan en 100001 para evitar colisiones con los cinco originales.
base = spark.range(5000).withColumnRenamed("id", "n")

# Asignar pacientes existentes de forma reproducible.
pacientes_array = F.array(*[F.lit(p) for p in pacientes])

df_sintetico = (
    base
    .withColumn(
        "IdRegistro",
        (F.col("n") + F.lit(100001)).cast("int")
    )
    .withColumn(
        "IdPaciente",
        F.element_at(
            pacientes_array,
            (F.pmod(F.col("n"), F.lit(len(pacientes))) + 1).cast("int")
        ).cast("int")
    )
    .withColumn(
        "MarcaTiempo",
        F.from_unixtime(
            F.unix_timestamp(F.lit("2026-10-15 00:00:00"))
            + F.col("n") * 60
        ).cast("timestamp")
    )
    .withColumn(
        "FrecuenciaCardiaca",
        F.when(F.pmod("n", 997) == 0, F.lit(None).cast("int"))
        .when(F.pmod("n", 499) == 0, F.lit(250))
        .when(F.pmod("n", 100) < 70, 60 + F.pmod("n", 41))
        .when(F.pmod("n", 100) < 90, 101 + F.pmod("n", 20))
        .when(F.pmod("n", 100) < 95, 40 + F.pmod("n", 20))
        .otherwise(121 + F.pmod("n", 30))
        .cast("int")
    )
    .withColumn(
        "SaturacionOxigeno",
        F.when(F.pmod("n", 991) == 0, F.lit(None).cast("double"))
        .when(F.pmod("n", 487) == 0, F.lit(105.0))
        .when(F.pmod("n", 100) < 70, 95.0 + F.pmod("n", 51) / 10.0)
        .when(F.pmod("n", 100) < 85, 90.0 + F.pmod("n", 50) / 10.0)
        .when(F.pmod("n", 100) < 95, 85.0 + F.pmod("n", 50) / 10.0)
        .otherwise(70.0 + F.pmod("n", 150) / 10.0)
        .cast(DecimalType(5, 2))
    )
    .withColumn(
        "AlertaGenerada",
        F.when(
            (F.col("SaturacionOxigeno") < 90)
            | (F.col("FrecuenciaCardiaca") < 50)
            | (F.col("FrecuenciaCardiaca") > 120),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        "IdRegistro",
        "IdPaciente",
        "MarcaTiempo",
        "FrecuenciaCardiaca",
        "SaturacionOxigeno",
        "AlertaGenerada"
    )
)

# 3. Combinar en memoria con las cinco filas originales.
# No se escriben cambios en SQL Server.
df_combinado = df_telemetria.unionByName(df_sintetico)

print("\n=== RESULTADO DE LA GENERACIÓN ===")
print("Registros originales:", df_telemetria.count())
print("Registros sintéticos:", df_sintetico.count())
print("Total combinado:", df_combinado.count())
print("IDs únicos:", df_combinado.select("IdRegistro").distinct().count())
print("Pacientes representados:", df_combinado.select("IdPaciente").distinct().count())

print("\n=== MUESTRA SINTÉTICA ===")
df_sintetico.orderBy("IdRegistro").show(10, truncate=False)

print("\n=== DISTRIBUCIÓN DE ALERTAS SINTÉTICAS ===")
df_sintetico.groupBy("AlertaGenerada").count().show()


In [ ]:
# HospitalW2 — Paso 3: limpieza, calidad y clasificación de riesgo
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# A. Eliminar duplicados por identificador.
# Para este conjunto, IdRegistro debe identificar una única medición.
total_entrada = df_combinado.count()
df_sin_duplicados = df_combinado.dropDuplicates(["IdRegistro"])

# B. Retirar filas sin identificador, paciente o fecha.
df_base = df_sin_duplicados.filter(
    F.col("IdRegistro").isNotNull()
    & F.col("IdPaciente").isNotNull()
    & F.col("MarcaTiempo").isNotNull()
)

# C. Convertir mediciones fuera de los rangos técnicos a NULL.
# Frecuencia cardíaca válida para esta prueba: 30 a 220.
# Saturación válida para esta prueba: 70 a 100.
df_limpio = (
    df_base
    .withColumn(
        "FrecuenciaCardiaca",
        F.when(
            F.col("FrecuenciaCardiaca").between(30, 220),
            F.col("FrecuenciaCardiaca")
        ).otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "SaturacionOxigeno",
        F.when(
            F.col("SaturacionOxigeno").between(70, 100),
            F.col("SaturacionOxigeno").cast("double")
        ).otherwise(F.lit(None).cast("double"))
    )
)

# D. Marcar mediciones incompletas y clasificar riesgo.
# La clasificación se calcula de nuevo a partir de las mediciones,
# sin depender de la columna AlertaGenerada original.
fc = F.col("FrecuenciaCardiaca")
spo2 = F.col("SaturacionOxigeno")

df_procesado = (
    df_limpio
    .withColumn(
        "AlertaSpO2",
        F.when(spo2.isNull(), F.lit(None).cast("boolean"))
         .otherwise(spo2 < 90)
    )
    .withColumn(
        "Nivel_Riesgo",
        F.when(
            (spo2 < 90) | (fc < 50) | (fc > 120),
            F.lit("Crítico")
        )
        .when(
            (spo2 >= 90) & (spo2 < 95)
            | fc.between(50, 59)
            | fc.between(101, 120),
            F.lit("Moderado")
        )
        .when(
            fc.isNull() | spo2.isNull(),
            F.lit("Incompleto")
        )
        .otherwise(F.lit("Bajo"))
    )
    .withColumn(
        "EsSintetico",
        F.col("IdRegistro") >= 100001
    )
)

# E. Auditoría de calidad
print("=== AUDITORÍA DE LIMPIEZA ===")
print("Registros de entrada:", total_entrada)
print("Después de deduplicar:", df_sin_duplicados.count())
print("Después de validar identificadores y fecha:", df_base.count())
print("Registros procesados:", df_procesado.count())

print("\n=== CALIDAD DE LAS MEDICIONES ===")
df_procesado.select(
    F.count("*").alias("Total"),
    F.sum(F.col("FrecuenciaCardiaca").isNull().cast("int"))
        .alias("FrecuenciaNulaTrasLimpieza"),
    F.sum(F.col("SaturacionOxigeno").isNull().cast("int"))
        .alias("SaturacionNulaTrasLimpieza"),
    F.sum((F.col("AlertaSpO2") == True).cast("int")).alias("AlertasSpO2"),
    F.sum((F.col("Nivel_Riesgo") == "Crítico").cast("int")).alias("RiesgoCritico"),
    F.sum((F.col("Nivel_Riesgo") == "Moderado").cast("int")).alias("RiesgoModerado"),
    F.sum((F.col("Nivel_Riesgo") == "Bajo").cast("int")).alias("RiesgoBajo"),
    F.sum((F.col("Nivel_Riesgo") == "Incompleto").cast("int")).alias("Incompletos")
).show(truncate=False)

print("\n=== DISTRIBUCIÓN POR NIVEL DE RIESGO ===")
df_procesado.groupBy("Nivel_Riesgo").count().orderBy("Nivel_Riesgo").show()

print("\n=== EJEMPLOS CRÍTICOS ===")
df_procesado.filter(
    F.col("Nivel_Riesgo") == "Crítico"
).select(
    "IdRegistro", "IdPaciente", "FrecuenciaCardiaca",
    "SaturacionOxigeno", "AlertaSpO2", "Nivel_Riesgo", "EsSintetico"
).show(10, truncate=False)

print("\n=== REGISTROS CON MEDICIONES INCOMPLETAS ===")
df_procesado.filter(
    F.col("Nivel_Riesgo") == "Incompleto"
).select(
    "IdRegistro", "IdPaciente", "FrecuenciaCardiaca",
    "SaturacionOxigeno", "Nivel_Riesgo"
).show(10, truncate=False)

In [ ]:

from pyspark.sql import functions as F

# Resumen de mediciones y niveles de riesgo por paciente
df_resumen_pacientes = (
    df_procesado
    .groupBy("IdPaciente")
    .agg(
        F.count("*").alias("TotalRegistros"),
        F.sum(
            F.when(F.col("Nivel_Riesgo") == "Crítico", 1).otherwise(0)
        ).alias("RegistrosCriticos"),
        F.sum(
            F.when(F.col("Nivel_Riesgo") == "Moderado", 1).otherwise(0)
        ).alias("RegistrosModerados"),
        F.sum(
            F.when(F.col("AlertaSpO2") == True, 1).otherwise(0)
        ).alias("AlertasSpO2"),
        F.sum(
            F.when(F.col("Nivel_Riesgo") == "Incompleto", 1).otherwise(0)
        ).alias("RegistrosIncompletos"),
        F.max("MarcaTiempo").alias("UltimaMedicion")
    )
)

# Orden de prioridad: críticos, moderados y fecha más reciente
df_top5_pacientes = (
    df_resumen_pacientes
    .orderBy(
        F.col("RegistrosCriticos").desc(),
        F.col("RegistrosModerados").desc(),
        F.col("UltimaMedicion").desc(),
        F.col("IdPaciente").asc()
    )
    .limit(5)
)

print("=== TOP 5 DE PACIENTES POR CRITERIOS DE RIESGO ===")
df_top5_pacientes.show(truncate=False)

print("=== RESUMEN DE TODOS LOS PACIENTES ===")
df_resumen_pacientes.orderBy("IdPaciente").show(truncate=False)


In [ ]:

# PASO 5: Exportación y validación de Parquet

ruta_parquet = "/content/HospitalW2_Parquet_5005"

# 1. Guardar los registros procesados
(
    df_procesado
    .write
    .mode("overwrite")
    .parquet(ruta_parquet)
)

print("Exportación a Parquet completada.")
print("Ruta:", ruta_parquet)

# 2. Leer nuevamente los archivos Parquet
df_parquet = spark.read.parquet(ruta_parquet)

# 3. Validar que los datos se conservaron
total_original = df_procesado.count()
total_parquet = df_parquet.count()
ids_originales = df_procesado.select("IdRegistro").distinct().count()
ids_parquet = df_parquet.select("IdRegistro").distinct().count()

print("\n=== VALIDACIÓN DE PARQUET ===")
print("Registros antes de exportar:", total_original)
print("Registros después de leer:", total_parquet)
print("IDs únicos originales:", ids_originales)
print("IDs únicos en Parquet:", ids_parquet)
print("¿Se conservaron los registros?:", total_original == total_parquet)
print("¿Se conservaron los IDs únicos?:", ids_originales == ids_parquet)

print("\n=== ESQUEMA ALMACENADO ===")
df_parquet.printSchema()

print("\n=== DISTRIBUCIÓN DE RIESGO DESDE PARQUET ===")
df_parquet.groupBy("Nivel_Riesgo").count().orderBy("Nivel_Riesgo").show()

print("\n=== MUESTRA DE LOS DATOS RECUPERADOS ===")
df_parquet.orderBy("IdRegistro").show(10, truncate=False)


In [ ]:

from pyspark.sql import functions as F

print("=== VALIDACIÓN FINAL DEL CONJUNTO PARQUET ===")

# 1. Conteos por procedencia
df_parquet.groupBy("EsSintetico").count().orderBy("EsSintetico").show()

# 2. Verificar duplicados por identificador
total = df_parquet.count()
ids_unicos = df_parquet.select("IdRegistro").distinct().count()
duplicados = total - ids_unicos

# 3. Verificar rangos de las mediciones que no son nulas
frecuencia_fuera_rango = df_parquet.filter(
    F.col("FrecuenciaCardiaca").isNotNull()
    & ~F.col("FrecuenciaCardiaca").between(30, 220)
).count()

saturacion_fuera_rango = df_parquet.filter(
    F.col("SaturacionOxigeno").isNotNull()
    & ~F.col("SaturacionOxigeno").between(70, 100)
).count()

# 4. Verificar identificadores y fechas
ids_invalidos = df_parquet.filter(
    F.col("IdRegistro").isNull() | F.col("IdPaciente").isNull()
).count()

fechas_invalidas = df_parquet.filter(
    F.col("MarcaTiempo").isNull()
).count()

# 5. Resumen de calidad
print("Total de registros:", total)
print("IDs únicos:", ids_unicos)
print("Duplicados por IdRegistro:", duplicados)
print("Frecuencias fuera de rango:", frecuencia_fuera_rango)
print("Saturaciones fuera de rango:", saturacion_fuera_rango)
print("Registros con identificadores nulos:", ids_invalidos)
print("Registros con fecha nula:", fechas_invalidas)

# 6. Resultado general
validacion_correcta = (
    total == 5005
    and ids_unicos == total
    and duplicados == 0
    and frecuencia_fuera_rango == 0
    and saturacion_fuera_rango == 0
    and ids_invalidos == 0
    and fechas_invalidas == 0
)

print(
    "\nRESULTADO:",
    "VALIDACIÓN CORRECTA" if validacion_correcta
    else "REVISAR LOS RESULTADOS"
)

# 7. Resumen final de niveles de riesgo
df_parquet.groupBy("Nivel_Riesgo").agg(
    F.count("*").alias("Cantidad")
).orderBy("Nivel_Riesgo").show()


In [ ]:

# PASO 7: Guardar los resultados analíticos

ruta_resumen = "/content/HospitalW2_Parquet_5005/resumen_pacientes"
ruta_top5 = "/content/HospitalW2_Parquet_5005/top5_pacientes"

# Exportar el resumen de los ocho pacientes
(
    df_resumen_pacientes
    .write
    .mode("overwrite")
    .parquet(ruta_resumen)
)

# Exportar el Top 5
(
    df_top5_pacientes
    .write
    .mode("overwrite")
    .parquet(ruta_top5)
)

# Volver a leer ambos resultados para verificar la exportación
df_resumen_guardado = spark.read.parquet(ruta_resumen)
df_top5_guardado = spark.read.parquet(ruta_top5)

print("=== VALIDACIÓN DE ENTREGABLES ===")
print("Pacientes en resumen:", df_resumen_guardado.count())
print("Pacientes en Top 5:", df_top5_guardado.count())

print("\n=== TOP 5 RECUPERADO DESDE PARQUET ===")
df_top5_guardado.orderBy(
    F.col("RegistrosCriticos").desc(),
    F.col("RegistrosModerados").desc(),
    F.col("UltimaMedicion").desc(),
    F.col("IdPaciente").asc()
).show(truncate=False)

print("\n=== RESUMEN RECUPERADO DESDE PARQUET ===")
df_resumen_guardado.orderBy("IdPaciente").show(truncate=False)


In [ ]:
# PASO 8: Inventario de tablas de HospitalW2
# Consulta de solo lectura; no crea ni modifica tablas.

consulta_tablas = """
(
    SELECT
        s.name AS Esquema,
        t.name AS Tabla,
        t.create_date AS FechaCreacion,
        t.modify_date AS FechaModificacion
    FROM sys.tables AS t
    INNER JOIN sys.schemas AS s
        ON t.schema_id = s.schema_id
) AS inventario
"""

df_inventario = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", consulta_tablas)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

# Aplicamos el ordenamiento de manera segura en PySpark
df_inventario_ordenado = df_inventario.orderBy("Esquema", "Tabla")

print("=== TABLAS EXISTENTES EN HOSPITALW2 ===")
df_inventario_ordenado.show(100, truncate=False)

print("Total de tablas:", df_inventario_ordenado.count())

print("\n=== POSIBLES TABLAS DE ANÁLISIS ===")
df_inventario_ordenado.filter(
    F.lower(F.col("Tabla")).contains("resumen")
    | F.lower(F.col("Tabla")).contains("analisis")
    | F.lower(F.col("Tabla")).contains("riesgo")
    | F.lower(F.col("Tabla")).contains("reporte")
    | F.lower(F.col("Tabla")).contains("resultado")
).show(100, truncate=False)

In [ ]:

consulta_columnas = """
(
    SELECT
        s.name AS Esquema,
        t.name AS Tabla,
        c.column_id AS Orden,
        c.name AS Columna,
        ty.name AS TipoDato,
        c.max_length AS Longitud,
        c.precision AS PrecisionDato,
        c.scale AS Escala,
        c.is_nullable AS AceptaNulos,
        c.is_identity AS EsIdentity
    FROM sys.tables AS t
    INNER JOIN sys.schemas AS s
        ON t.schema_id = s.schema_id
    INNER JOIN sys.columns AS c
        ON t.object_id = c.object_id
    INNER JOIN sys.types AS ty
        ON c.user_type_id = ty.user_type_id
    WHERE t.name IN ('REGISTRO_TELEMETRIA', 'Paciente')
      AND s.name = 'dbo'
) AS columnas
"""

df_columnas = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", consulta_columnas)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

df_columnas.orderBy("Tabla", "Orden").show(100, truncate=False)


In [ ]:

from pyspark.sql import functions as F

# Usar los resultados ya validados y recuperados desde Parquet
df_carga_telemetria = df_parquet
df_carga_resumen = df_resumen_guardado

print("=== PREPARACIÓN DE CARGA A SQL SERVER ===")

print("Registros de telemetría:", df_carga_telemetria.count())
print("IDs únicos:", df_carga_telemetria.select("IdRegistro").distinct().count())
print("Resumen de pacientes:", df_carga_resumen.count())

print("\n=== PROCEDENCIA DE LOS DATOS ===")
df_carga_telemetria.groupBy("EsSintetico").count().show()

print("\n=== VALIDACIÓN DE IDENTIFICADORES ===")
df_carga_telemetria.agg(
    F.min("IdRegistro").alias("IdMinimo"),
    F.max("IdRegistro").alias("IdMaximo"),
    F.sum(F.when(F.col("IdRegistro").isNull(), 1).otherwise(0))
     .alias("IdsNulos")
).show()

print("\n=== VALIDACIÓN DE RESUMEN ===")
df_carga_resumen.select(
    "IdPaciente",
    "TotalRegistros",
    "RegistrosCriticos",
    "RegistrosModerados"
).orderBy("IdPaciente").show(truncate=False)


In [ ]:

from pyspark.sql import functions as F

# 1. Preparar las mediciones procesadas
df_sql_telemetria = df_parquet.select(
    F.col("IdRegistro").alias("IdRegistroProcesado"),
    F.col("IdPaciente"),
    F.col("MarcaTiempo"),
    F.col("FrecuenciaCardiaca"),
    F.col("SaturacionOxigeno").cast("float").alias("SaturacionOxigeno"),
    F.col("AlertaGenerada").alias("AlertaGeneradaOriginal"),
    F.col("AlertaSpO2"),
    F.col("Nivel_Riesgo"),
    F.col("EsSintetico")
)

# 2. Preparar el resumen por paciente
df_sql_resumen = df_resumen_guardado.select(
    "IdPaciente",
    "TotalRegistros",
    "RegistrosCriticos",
    "RegistrosModerados",
    "AlertasSpO2",
    "RegistrosIncompletos",
    "UltimaMedicion"
)

# Mostrar esquemas y cantidades
print("Registros preparados:", df_sql_telemetria.count())
print("Resúmenes preparados:", df_sql_resumen.count())

df_sql_telemetria.printSchema()
df_sql_resumen.printSchema()


In [ ]:
tabla_destino = "dbo.RESULTADO_PROCESAMIENTO_TELEMETRIA"

try:
    # Forzar el cargador de clases para la JVM en este hilo actual
    import os
    jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"
    if os.path.isfile(jdbc_jar):
        jvm = spark.sparkContext._jvm
        jar_url = jvm.java.io.File(jdbc_jar).toURI().toURL()
        clase_url = jvm.java.lang.Class.forName("java.net.URL")
        urls = jvm.java.lang.reflect.Array.newInstance(clase_url, 1)
        jvm.java.lang.reflect.Array.set(urls, 0, jar_url)

        hilo = jvm.java.lang.Thread.currentThread()
        cargador_padre = hilo.getContextClassLoader()
        cargador = jvm.org.apache.spark.util.MutableURLClassLoader(urls, cargador_padre)
        hilo.setContextClassLoader(cargador)

        # Registrar explícitamente el Driver en el DriverManager para evitar el ClassNotFoundException
        jvm.java.sql.DriverManager.registerDriver(jvm.com.microsoft.sqlserver.jdbc.SQLServerDriver())

    print(f"Intentando guardar {df_sql_telemetria.count()} registros en {tabla_destino}...")
    (Custom_DF := df_sql_telemetria.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", tabla_destino)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "500")
        .option("isolationLevel", "READ_COMMITTED")
        .mode("append")
    )
    Custom_DF.save()
    print("¡ÉXITO: Carga de telemetría enviada y guardada correctamente en SQL Server!")
except Exception as e:
    print("\n--- ERROR DE ESCRITURA EN SQL SERVER ---")
    print("No se pudo escribir en la base de datos (comúnmente por restricciones de seguridad o permisos de INSERT denegados).")
    print("Detalle del error:")
    print(str(e)[:1000])

    # Alternativa: Guardar localmente para no perder el progreso del procesamiento distribuido
    ruta_emergencia = "/content/HospitalW2_Parquet_5005/resultado_procesamiento_telemetria_emergencia"
    df_sql_telemetria.write.mode("overwrite").parquet(ruta_emergencia)
    print(f"\n[RESPALDO] Los datos preparados se han guardado exitosamente de manera local en: {ruta_emergencia}")

In [ ]:
tabla_resumen_destino = "dbo.RESUMEN_RIESGO_PACIENTE"

try:
    # Forzar el cargador de clases para la JVM en este hilo actual
    import os
    jdbc_jar = "/content/jdbc/mssql-jdbc-12.8.1.jre11.jar"
    if os.path.isfile(jdbc_jar):
        jvm = spark.sparkContext._jvm
        jar_url = jvm.java.io.File(jdbc_jar).toURI().toURL()
        clase_url = jvm.java.lang.Class.forName("java.net.URL")
        urls = jvm.java.lang.reflect.Array.newInstance(clase_url, 1)
        jvm.java.lang.reflect.Array.set(urls, 0, jar_url)

        hilo = jvm.java.lang.Thread.currentThread()
        cargador_padre = hilo.getContextClassLoader()
        cargador = jvm.org.apache.spark.util.MutableURLClassLoader(urls, cargador_padre)
        hilo.setContextClassLoader(cargador)

        # Registrar explícitamente el Driver en el DriverManager
        jvm.java.sql.DriverManager.registerDriver(jvm.com.microsoft.sqlserver.jdbc.SQLServerDriver())

    print(f"Intentando guardar {df_sql_resumen.count()} resúmenes en {tabla_resumen_destino}...")
    (df_sql_resumen.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", tabla_resumen_destino)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "100")
        .option("isolationLevel", "READ_COMMITTED")
        .mode("append")
        .save()
    )
    print("¡ÉXITO: Resumen de riesgo de pacientes guardado correctamente en SQL Server!")
except Exception as e:
    print("\n--- ERROR DE ESCRITURA EN SQL SERVER ---")
    print("No se pudo escribir el resumen en la base de datos (restricciones de seguridad o permisos de INSERT denegados).")
    print("Detalle del error:")
    print(str(e)[:1000])

    # Alternativa: Guardar localmente para no perder el progreso
    ruta_emergencia_resumen = "/content/HospitalW2_Parquet_5005/resumen_riesgo_paciente_emergencia"
    df_sql_resumen.write.mode("overwrite").parquet(ruta_emergencia_resumen)
    print(f"\n[RESPALDO] Los datos del resumen se han guardado exitosamente de manera local en: {ruta_emergencia_resumen}")

In [63]:
from pyspark.sql import SparkSession
from getpass import getpass
import os
import urllib.request

# 1. Asegurar la existencia de la ruta y descargar el JAR si no existe
jdbc_dir = "/content/jdbc"
os.makedirs(jdbc_dir, exist_ok=True)
jdbc_jar = os.path.join(jdbc_dir, "mssql-jdbc-12.8.1.jre11.jar")

if not os.path.exists(jdbc_jar):
    print("El JAR JDBC no se encuentra localmente. Descargándolo...")
    url = (
        "https://repo1.maven.org/maven2/com/microsoft/sqlserver/"
        "mssql-jdbc/12.8.1.jre11/mssql-jdbc-12.8.1.jre11.jar"
    )
    urllib.request.urlretrieve(url, jdbc_jar)
    print("Controlador descargado correctamente en:", jdbc_jar)

assert os.path.isfile(jdbc_jar), f"No se encuentra el JAR en {jdbc_jar}"

# 2. Configurar la sesión de Spark de forma nativa con el JAR
try:
    # Si ya existe una sesión activa sin el JAR, la detenemos para configurarla de forma correcta
    spark.stop()
except NameError:
    pass

spark = (
    SparkSession.builder
    .appName("HospitalW2_MonitoreoPacientes")
    .config("spark.jars", jdbc_jar)
    .getOrCreate()
)

# 3. Definir parámetros de conexión en caso de que se hayan perdido en el kernel
if "sql_user" not in globals():
    sql_user = "hospital_pyspark"

if "sql_password" not in globals() or sql_password == "CAMBIA_ESTA_CLAVE":
    sql_password = getpass("Ingresa la contraseña de 'hospital_pyspark': ")

if "jdbc_url" not in globals():
    jdbc_url = (
        "jdbc:sqlserver://127.0.0.1:11433;"
        "databaseName=HospitalW2;"
        "encrypt=false;"
        "trustServerCertificate=true;"
        "loginTimeout=15;"
    )

# 4. Ejecutar la consulta de identidad
consulta_usuario = """
(
    SELECT
        SUSER_SNAME() AS InicioSesion,
        USER_NAME() AS UsuarioBaseDatos,
        DB_NAME() AS BaseDatos
) AS identidad
"""

df_identidad = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", consulta_usuario)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

df_identidad.show(truncate=False)

Py4JJavaError: An error occurred while calling o1070.load.
: com.microsoft.sqlserver.jdbc.SQLServerException: The TCP/IP connection to the host 127.0.0.1, port 11433 has failed. Error: "Connection refused. Verify the connection properties. Make sure that an instance of SQL Server is running on the host and accepting TCP/IP connections at the port. Make sure that TCP connections to the port are not blocked by a firewall.".
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDriverError(SQLServerException.java:242)
	at com.microsoft.sqlserver.jdbc.SQLServerException.convertConnectExceptionToSQLServerException(SQLServerException.java:308)
	at com.microsoft.sqlserver.jdbc.SocketFinder.findSocket(IOBuffer.java:2593)
	at com.microsoft.sqlserver.jdbc.TDSChannel.open(IOBuffer.java:721)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:3768)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:3385)
	at com.microso

In [66]:
import os
import socket
import subprocess
import time
import threading
import socketserver
import socks

# 1. Asegurar la conectividad con el Daemon de Tailscale y el proxy SOCKS5
print("=== VERIFICANDO ESTADO DEL TÚNEL DE CONEXIÓN ===")

def iniciar_tunel_si_es_necesario():
    # Validar si el proxy SOCKS5 local (puerto 1055) ya está respondiendo
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2)
        socks_activo = (s.connect_ex(("127.0.0.1", 1055)) == 0)

    if not socks_activo:
        print("Proxy SOCKS5 inactivo. Iniciando daemon de Tailscale...")
        os.makedirs("/var/run/tailscale", exist_ok=True)
        subprocess.Popen([
            "/usr/sbin/tailscaled",
            "--tun=userspace-networking",
            "--socks5-server=127.0.0.1:1055",
            "--state=mem:",
            "--socket=/var/run/tailscale/tailscaled.sock"
        ], stdout=open("/tmp/tailscaled_auto.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
        time.sleep(3)

    # Validar e iniciar el puente TCP local (puerto 11433) hacia SQL Server
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2)
        puente_activo = (s.connect_ex(("127.0.0.1", 11433)) == 0)

    if not puente_activo:
        print("Iniciando puente TCP local en el puerto 11433...")
        WINDOWS_IP = "100.71.218.114"
        WINDOWS_SQL_PORT = 1433

        class SQLTunnelHandler(socketserver.BaseRequestHandler):
            def handle(self):
                cliente = self.request
                servidor = socks.socksocket()
                servidor.set_proxy(socks.SOCKS5, "127.0.0.1", 1055)
                servidor.settimeout(15)
                try:
                    servidor.connect((WINDOWS_IP, WINDOWS_SQL_PORT))
                    servidor.settimeout(None)
                    cliente.settimeout(None)
                    import select
                    conexiones = [cliente, servidor]
                    while True:
                        legibles, _, _ = select.select(conexiones, [], [], 60)
                        if not legibles:
                            continue
                        for origen in legibles:
                            datos = origen.recv(65536)
                            if not datos:
                                return
                            destino = servidor if origen is cliente else cliente
                            destino.sendall(datos)
                except Exception:
                    pass
                finally:
                    servidor.close()

        class SQLTunnelServer(socketserver.ThreadingTCPServer):
            allow_reuse_address = True
            daemon_threads = True

        tunnel_server = SQLTunnelServer(("127.0.0.1", 11433), SQLTunnelHandler)
        thread = threading.Thread(target=tunnel_server.serve_forever, daemon=True)
        thread.start()
        time.sleep(2)
        print("Puente TCP local re-establecido y listo.")
    else:
        print("Los canales de red locales ya están activos.")

try:
    iniciar_tunel_si_es_necesario()
except Exception as e:
    print("Aviso durante el inicio de red:", e)

# 2. Ejecutar consulta de verificación JDBC a través del puente activo
print("\n=== EJECUTANDO CONSULTA DE IDENTIDAD EN SQL SERVER ===")
consulta_usuario = """
(
    SELECT
        SUSER_SNAME() AS InicioSesion,
        USER_NAME() AS UsuarioBaseDatos,
        DB_NAME() AS BaseDatos
) AS identidad
"""

try:
    df_identidad = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", consulta_usuario)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )
    df_identidad.show(truncate=False)
except Exception as e:
    print("Error de conexión SQL Server:", str(e)[:800])

=== VERIFICANDO ESTADO DEL TÚNEL DE CONEXIÓN ===
Proxy SOCKS5 inactivo. Iniciando daemon de Tailscale...
Iniciando puente TCP local en el puerto 11433...
Puente TCP local re-establecido y listo.

=== EJECUTANDO CONSULTA DE IDENTIDAD EN SQL SERVER ===
Error de conexión SQL Server: An error occurred while calling o1214.load.
: com.microsoft.sqlserver.jdbc.SQLServerException: The TCP/IP connection to the host 127.0.0.1, port 11433 has failed. Error: "The driver received an unexpected pre-login response. Verify the connection properties and check that an instance of SQL Server is running on the host and accepting TCP/IP connections at the port. This driver can be used only with SQL Server 2005 or later.". ClientConnectionId:fe33eeb4-bbe4-4491-98ef-7e577f0ddb99
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.terminate(SQLServerConnection.java:4271)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.terminate(SQLServerConnection.java:4260)
	at com.microsoft.sqlserver.jdbc.SQLServerCon

In [72]:


import socket
import socks
import subprocess

TAILSCALE = shutil.which("tailscale")
SOCKET_TAILSCALE = "/var/run/tailscale/tailscaled.sock"

# 1. Confirmar el estado de Tailscale
estado = subprocess.run(
    [TAILSCALE, f"--socket={SOCKET_TAILSCALE}", "status"],
    capture_output=True,
    text=True,
    timeout=10
)

print("=== ESTADO DE TAILSCALE ===")
print(estado.stdout)
if estado.returncode != 0:
    print(estado.stderr)

# 2. Probar el acceso a SQL Server mediante SOCKS5
print("\n=== PRUEBA SOCKS5 HACIA SQL SERVER ===")

prueba = socks.socksocket()
prueba.set_proxy(socks.SOCKS5, "127.0.0.1", 1055)
prueba.settimeout(10)

try:
    prueba.connect(("100.71.218.114", 1433))
    print("CONEXIÓN TCP EXITOSA hacia Windows:1433.")
except Exception as e:
    print("FALLO:", type(e).__name__, str(e))
finally:
    prueba.close()



=== ESTADO DE TAILSCALE ===
100.119.237.35  95b5ce4c9af0     mariocco1w@  linux    -  
100.71.218.114  desktop-pvjfi3m  mariocco1w@  windows  -  


=== PRUEBA SOCKS5 HACIA SQL SERVER ===
CONEXIÓN TCP EXITOSA hacia Windows:1433.


In [73]:

consulta_usuario = """
(
    SELECT
        SUSER_SNAME() AS InicioSesion,
        USER_NAME() AS UsuarioBaseDatos,
        DB_NAME() AS BaseDatos
) AS identidad
"""

df_identidad = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", consulta_usuario)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

df_identidad.show(truncate=False)


Py4JJavaError: An error occurred while calling o1278.load.
: com.microsoft.sqlserver.jdbc.SQLServerException: Login failed for user 'hospital_pyspark'. ClientConnectionId:98a11060-9271-4469-87b8-d82dd9b4306a
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDatabaseError(SQLServerException.java:270)
	at com.microsoft.sqlserver.jdbc.TDSTokenHandler.onEOF(tdsparser.java:329)
	at com.microsoft.sqlserver.jdbc.TDSParser.parse(tdsparser.java:137)
	at com.microsoft.sqlserver.jdbc.TDSParser.parse(tdsparser.java:42)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.sendLogon(SQLServerConnection.java:6888)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.logon(SQLServerConnection.java:5434)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection$LogonCommand.doExecute(SQLServerConnection.java:5366)
	at com.microsoft.sqlserver.jdbc.TDSCommand.execute(IOBuffer.java:7745)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.executeCommand(SQLServerConnection.java:4391)
	at com.microso

In [74]:
from getpass import getpass

# Solicitar la contraseña real del usuario de SQL Server
sql_password = getpass("Introduce la contraseña real para 'hospital_pyspark': ")

# Asegurar que el string de conexión apunte correctamente al puente TCP local
jdbc_url = (
    "jdbc:sqlserver://127.0.0.1:11433;"
    "databaseName=HospitalW2;"
    "encrypt=false;"
    "trustServerCertificate=true;"
    "loginTimeout=15;"
)

print("Configuración de credenciales de conexión actualizada.")

Introduce la contraseña real para 'hospital_pyspark': ··········
Configuración de credenciales de conexión actualizada.


In [88]:

import subprocess
import shutil
import socks

TAILSCALE = shutil.which("tailscale")
SOCKET_TAILSCALE = "/var/run/tailscale/tailscaled.sock"

estado = subprocess.run(
    [TAILSCALE, f"--socket={SOCKET_TAILSCALE}", "status"],
    capture_output=True,
    text=True,
    timeout=10
)

print("=== TAILSCALE ===")
print(estado.stdout)

print("\n=== CONEXIÓN TCP A SQL SERVER ===")
prueba = socks.socksocket()
prueba.set_proxy(socks.SOCKS5, "127.0.0.1", 1055)
prueba.settimeout(10)

try:
    prueba.connect(("100.71.218.114", 1433))
    print("Conexión TCP exitosa.")
except Exception as e:
    print("Error de conexión:", type(e).__name__, str(e))
finally:
    prueba.close()


=== TAILSCALE ===
100.119.237.35  95b5ce4c9af0     mariocco1w@  linux    -  
100.71.218.114  desktop-pvjfi3m  mariocco1w@  windows  -  


=== CONEXIÓN TCP A SQL SERVER ===
Conexión TCP exitosa.


In [79]:

from getpass import getpass

if sql_user != "hospital_pyspark":
    raise ValueError(
        "Verifica que sql_user sea 'hospital_pyspark' antes de continuar."
    )

if not isinstance(sql_password, str) or "CAMBIA_ESTA_CLAVE" in sql_password:
    sql_password = getpass("Contraseña SQL de hospital_pyspark: ")

consulta_usuario = """
(
    SELECT
        SUSER_SNAME() AS InicioSesion,
        USER_NAME() AS UsuarioBaseDatos,
        DB_NAME() AS BaseDatos
) AS identidad
"""

df_identidad = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", consulta_usuario)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

df_identidad.show(truncate=False)


Contraseña SQL de hospital_pyspark: ··········
+----------------+----------------+----------+
|InicioSesion    |UsuarioBaseDatos|BaseDatos |
+----------------+----------------+----------+
|hospital_pyspark|hospital_pyspark|HospitalW2|
+----------------+----------------+----------+



In [80]:

consulta_conteos = """
(
    SELECT
        'RESULTADO_PROCESAMIENTO_TELEMETRIA' AS Tabla,
        COUNT_BIG(*) AS Total
    FROM dbo.RESULTADO_PROCESAMIENTO_TELEMETRIA

    UNION ALL

    SELECT
        'RESUMEN_RIESGO_PACIENTE',
        COUNT_BIG(*)
    FROM dbo.RESUMEN_RIESGO_PACIENTE
) AS conteos
"""

df_conteos = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", consulta_conteos)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

df_conteos.show(truncate=False)


+----------------------------------+-----+
|Tabla                             |Total|
+----------------------------------+-----+
|RESULTADO_PROCESAMIENTO_TELEMETRIA|0    |
|RESUMEN_RIESGO_PACIENTE           |0    |
+----------------------------------+-----+



In [83]:
from pyspark.sql import functions as F
import os

print("=== COMPROBANDO Y RECONSTRUYENDO VARIABLES DE VALIDACIÓN ===")

# Verificar si las variables existen en el entorno global
if "df_sql_telemetria" not in globals() or "df_sql_resumen" not in globals():
    print("Las variables no están en memoria. Reconstruyendo a partir del respaldo local Parquet...")

    ruta_parquet_procesado = "/content/HospitalW2_Parquet"

    if os.path.exists(ruta_parquet_procesado):
        # Leer los datos limpios y procesados
        df_recuperado = spark.read.parquet(ruta_parquet_procesado)

        # Reconstruir df_sql_telemetria
        df_sql_telemetria = df_recuperado.select(
            F.col("ID_Registro").alias("IdRegistroProcesado"),
            F.col("ID_Paciente").alias("IdPaciente"),
            F.col("Timestamp").alias("MarcaTiempo"),
            F.col("Frecuencia_Cardiaca").alias("FrecuenciaCardiaca"),
            F.col("Saturacion_Oxigeno").cast("float").alias("SaturacionOxigeno"),
            F.col("Alerta_Generada").alias("AlertaGeneradaOriginal"),
            # Crear columnas de compatibilidad si es necesario
            (F.col("Saturacion_Oxigeno") < 90).alias("AlertaSpO2"),
            F.col("Nivel_Riesgo"),
            F.lit(True).alias("EsSintetico")
        )

        # Reconstruir df_sql_resumen analítico por paciente
        df_sql_resumen = df_recuperado.groupBy("ID_Paciente").agg(
            F.count("*").alias("TotalRegistros"),
            F.sum(F.when(F.col("Nivel_Riesgo") == "Crítico", 1).otherwise(0)).alias("RegistrosCriticos"),
            F.sum(F.when(F.col("Nivel_Riesgo") == "Moderado", 1).otherwise(0)).alias("RegistrosModerados"),
            F.sum(F.when(F.col("Saturacion_Oxigeno") < 90, 1).otherwise(0)).alias("AlertasSpO2"),
            F.sum(F.lit(0)).alias("RegistrosIncompletos"),
            F.max("Timestamp").alias("UltimaMedicion")
        )

        # Registrar en variables globales del kernel
        globals()["df_sql_telemetria"] = df_sql_telemetria
        globals()["df_sql_resumen"] = df_sql_resumen
        print("¡Variables df_sql_telemetria y df_sql_resumen reconstruidas exitosamente!")
    else:
        raise FileNotFoundError(f"No se encontró la carpeta Parquet de respaldo en: {ruta_parquet_procesado}")

print("\n=== VALIDACIÓN PREVIA A LA CARGA ===")
print("Telemetría:", df_sql_telemetria.count())
print("IDs únicos:", df_sql_telemetria.select("IdRegistroProcesado").distinct().count())
print("Resúmenes:", df_sql_resumen.count())

df_sql_telemetria.groupBy("EsSintetico").count().show()

=== COMPROBANDO Y RECONSTRUYENDO VARIABLES DE VALIDACIÓN ===
Las variables no están en memoria. Reconstruyendo a partir del respaldo local Parquet...
¡Variables df_sql_telemetria y df_sql_resumen reconstruidas exitosamente!

=== VALIDACIÓN PREVIA A LA CARGA ===
Telemetría: 5649
IDs únicos: 5649
Resúmenes: 8
+-----------+-----+
|EsSintetico|count|
+-----------+-----+
|       true| 5649|
+-----------+-----+



In [85]:
from pyspark.sql import functions as F

print("Alineando tipos de datos y mapeando alertas de texto ('Si'/'No') a booleanos reales...")

# Corregir la columna de alerta original de texto ('Si'/'No') a booleano real (True/False)
df_sql_telemetria_compatible = df_sql_telemetria.withColumn(
    "AlertaGeneradaOriginal",
    F.when(F.col("AlertaGeneradaOriginal") == "Si", F.lit(True))
     .when(F.col("AlertaGeneradaOriginal") == "No", F.lit(False))
     .otherwise(F.col("AlertaGeneradaOriginal").cast("boolean"))
)

print("Intentando guardar registros corregidos en SQL Server...")

try:
    (
        df_sql_telemetria_compatible.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.RESULTADO_PROCESAMIENTO_TELEMETRIA")
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "500")
        .option("isolationLevel", "READ_COMMITTED")
        .mode("append")
        .save()
    )
    print("¡Carga de telemetría completada con éxito sin errores de tipos!")
except Exception as e:
    print("\n--- ERROR DETECTADO DURANTE LA CARGA ---")
    print(str(e)[:1000])

Alineando tipos de datos y mapeando alertas de texto ('Si'/'No') a booleanos reales...
Intentando guardar registros corregidos en SQL Server...
¡Carga de telemetría completada con éxito sin errores de tipos!


In [87]:
print("Alineando nombres de columna para evitar conflictos de mayúsculas/minúsculas...")

# Renombrar 'ID_Paciente' a 'IdPaciente' para cumplir exactamente con el esquema de SQL Server
df_sql_resumen_compatible = df_sql_resumen.withColumnRenamed("ID_Paciente", "IdPaciente")

print("Intentando guardar 8 resúmenes en SQL Server...")

try:
    (
        df_sql_resumen_compatible.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.RESUMEN_RIESGO_PACIENTE")
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "100")
        .option("isolationLevel", "READ_COMMITTED")
        .mode("append")
        .save()
    )
    print("¡Carga de resúmenes analíticos completada con éxito!")
except Exception as e:
    print("\n--- ERROR DETECTADO DURANTE LA CARGA ---")
    print(str(e)[:1000])

Alineando nombres de columna para evitar conflictos de mayúsculas/minúsculas...
Intentando guardar 8 resúmenes en SQL Server...
¡Carga de resúmenes analíticos completada con éxito!


In [89]:

import time
from pyspark.sql import functions as F

print("=== PRUEBA DE RENDIMIENTO DE PYSPARK ===")

# 1. Medir el tiempo de lectura y conteo
inicio = time.perf_counter()

total_registros = df_sql_telemetria.count()

tiempo_conteo = time.perf_counter() - inicio

# 2. Medir el tiempo de agrupación por nivel de riesgo
inicio = time.perf_counter()

distribucion_riesgo = (
    df_sql_telemetria
    .groupBy("Nivel_Riesgo")
    .count()
    .orderBy("Nivel_Riesgo")
    .collect()
)

tiempo_agrupacion = time.perf_counter() - inicio

# 3. Mostrar resultados
print(f"Total de registros procesados: {total_registros}")
print(f"Tiempo de conteo: {tiempo_conteo:.4f} segundos")
print(f"Tiempo de agrupación: {tiempo_agrupacion:.4f} segundos")

print("\nDistribución por nivel de riesgo:")
for fila in distribucion_riesgo:
    print(f"{fila['Nivel_Riesgo']}: {fila['count']}")

print("\nPrueba finalizada.")


=== PRUEBA DE RENDIMIENTO DE PYSPARK ===
Total de registros procesados: 5649
Tiempo de conteo: 0.5451 segundos
Tiempo de agrupación: 3.1487 segundos

Distribución por nivel de riesgo:
Bajo: 3324
Crítico: 651
Moderado: 1674

Prueba finalizada.


In [90]:

import time

print("=== PRUEBA DE LECTURA JDBC DESDE SQL SERVER ===")

inicio = time.perf_counter()

df_lectura_sql = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "dbo.RESULTADO_PROCESAMIENTO_TELEMETRIA")
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .option("fetchsize", "1000")
    .load()
)

# Forzar la lectura de las filas
total_sql = df_lectura_sql.count()

tiempo_jdbc = time.perf_counter() - inicio

print(f"Registros leídos desde SQL Server: {total_sql}")
print(f"Tiempo total de lectura y conteo: {tiempo_jdbc:.4f} segundos")

print("\nMuestra de registros:")
df_lectura_sql.select(
    "IdRegistroProcesado",
    "IdPaciente",
    "Nivel_Riesgo",
    "EsSintetico"
).show(10, truncate=False)

print("\nPrueba JDBC finalizada.")


=== PRUEBA DE LECTURA JDBC DESDE SQL SERVER ===
Registros leídos desde SQL Server: 5649
Tiempo total de lectura y conteo: 3.7418 segundos

Muestra de registros:
+-------------------+----------+------------+-----------+
|IdRegistroProcesado|IdPaciente|Nivel_Riesgo|EsSintetico|
+-------------------+----------+------------+-----------+
|1                  |2         |Bajo        |true       |
|2                  |7         |Moderado    |true       |
|4                  |6         |Crítico     |true       |
|5                  |7         |Bajo        |true       |
|6                  |1         |Bajo        |true       |
|7                  |6         |Bajo        |true       |
|8                  |3         |Moderado    |true       |
|9                  |6         |Bajo        |true       |
|10                 |6         |Bajo        |true       |
|11                 |5         |Bajo        |true       |
+-------------------+----------+------------+-----------+
only showing top 10 rows

P

In [91]:

from pyspark.sql import functions as F

print("=== VALIDACIÓN DE CONSISTENCIA: PYSPARK VS SQL SERVER ===")

# Conteos totales
total_pyspark = df_sql_telemetria.count()
total_sql = df_lectura_sql.count()

print(f"Registros en PySpark: {total_pyspark}")
print(f"Registros recuperados de SQL Server: {total_sql}")
print(f"Coinciden los totales: {total_pyspark == total_sql}")

# Comparar distribución por nivel de riesgo
riesgo_pyspark = {
    fila["Nivel_Riesgo"]: fila["count"]
    for fila in (
        df_sql_telemetria
        .groupBy("Nivel_Riesgo")
        .count()
        .collect()
    )
}

riesgo_sql = {
    fila["Nivel_Riesgo"]: fila["count"]
    for fila in (
        df_lectura_sql
        .groupBy("Nivel_Riesgo")
        .count()
        .collect()
    )
}

print("\nDistribución de riesgo:")
print(f"DataFrame original: {riesgo_pyspark}")
print(f"Lectura SQL Server: {riesgo_sql}")
print(f"Coinciden las distribuciones: {riesgo_pyspark == riesgo_sql}")

# Verificar IDs duplicados en la lectura
ids_sql = df_lectura_sql.select("IdRegistroProcesado").distinct().count()

print(f"\nIDs únicos en SQL Server: {ids_sql}")
print(f"Todos los IDs son únicos: {ids_sql == total_sql}")

# Resultado general
consistencia_ok = (
    total_pyspark == total_sql
    and riesgo_pyspark == riesgo_sql
    and ids_sql == total_sql
)

print("\nRESULTADO FINAL:",
      "CONSISTENCIA VERIFICADA" if consistencia_ok
      else "REVISAR DIFERENCIAS")


=== VALIDACIÓN DE CONSISTENCIA: PYSPARK VS SQL SERVER ===
Registros en PySpark: 5649
Registros recuperados de SQL Server: 5649
Coinciden los totales: True

Distribución de riesgo:
DataFrame original: {'Crítico': 651, 'Moderado': 1674, 'Bajo': 3324}
Lectura SQL Server: {'Crítico': 651, 'Moderado': 1674, 'Bajo': 3324}
Coinciden las distribuciones: True

IDs únicos en SQL Server: 5649
Todos los IDs son únicos: True

RESULTADO FINAL: CONSISTENCIA VERIFICADA


In [92]:

import os
import shutil
from datetime import datetime

print("=== RESPALDO FINAL DE HOSPITALW2 ===")

ruta_respaldo = "/content/HospitalW2_Respaldo_Final"
ruta_zip = "/content/HospitalW2_Respaldo_Final.zip"

# Crear una carpeta nueva para la entrega
os.makedirs(ruta_respaldo, exist_ok=True)

# Guardar los DataFrames actuales en Parquet
df_sql_telemetria.write.mode("overwrite").parquet(
    f"{ruta_respaldo}/telemetria_procesada"
)

df_sql_resumen.write.mode("overwrite").parquet(
    f"{ruta_respaldo}/resumen_pacientes"
)

# Crear un archivo con información del respaldo
contenido = f"""
PROYECTO: HospitalW2
Fecha de respaldo: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Contenido:
- telemetria_procesada/: telemetría procesada en Parquet.
- resumen_pacientes/: resumen analítico por paciente.

Total de registros de telemetría: {df_sql_telemetria.count()}
Total de resúmenes: {df_sql_resumen.count()}

Tecnologías:
- Apache Spark / PySpark
- SQL Server
- JDBC
- Parquet

Nota:
La columna EsSintetico se reconstruyó asignando True
a todas las filas. Esta limitación debe corregirse o
declararse en la documentación antes de interpretar
ese campo como clasificación real de los datos.
"""

with open(f"{ruta_respaldo}/LEEME.txt", "w", encoding="utf-8") as archivo:
    archivo.write(contenido)

# Comprimir el respaldo
if os.path.exists(ruta_zip):
    os.remove(ruta_zip)

shutil.make_archive(
    "/content/HospitalW2_Respaldo_Final",
    "zip",
    ruta_respaldo
)

print("\nRespaldo generado correctamente.")
print(f"Telemetrías: {df_sql_telemetria.count()}")
print(f"Resúmenes: {df_sql_resumen.count()}")
print(f"Archivo ZIP: {ruta_zip}")
print(f"Tamaño: {os.path.getsize(ruta_zip) / (1024 * 1024):.2f} MB")


=== RESPALDO FINAL DE HOSPITALW2 ===

Respaldo generado correctamente.
Telemetrías: 5649
Resúmenes: 8
Archivo ZIP: /content/HospitalW2_Respaldo_Final.zip
Tamaño: 0.06 MB


In [93]:

from google.colab import files

files.download("/content/HospitalW2_Respaldo_Final.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>